In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from glob import glob
import os
from pyproj import CRS, Transformer
import datetime
from geopy.distance import geodesic

In [ ]:
def lonlat_to_xy(lon, lat):
    crs1 = CRS.from_epsg(4326) # from WGS 84
    crs2 = CRS.from_epsg(3031) # to south polar stereographic at -71° S

    transformer = Transformer.from_crs(crs1, crs2, always_xy=True)
    x,y = transformer.transform(lon, lat)
    return x,y

In [ ]:
# point to the CSRS-processed data
fold = 'processed_rinex'
years = glob(os.path.join(fold ,'2*'))
years.sort()

In [ ]:
### load in all the data; two parallel vectors: one with a dataframe for the data of data 
### and one with the date of the data (for easy searching)
high_rez = []
dates = []
for yy in years:
    print('working on ' + yy)
    files = glob(os.path.join(yy, '*.pos'))
    files.sort()
    for this_file in files:
        data = pd.read_table(this_file, header = 4, sep=r'\s+')
        data['lat_decimal'] = data['LATDD']-data['LATMN']/60-data['LATSS']/3600
        data['lon_decimal'] = data['LONDD']+data['LONMN']/60+data['LONSS']/3600
        data['x'], data['y'] = lonlat_to_xy(data['lon_decimal'], data['lat_decimal'])

        # make datetime64 column
        times = data['YEAR-MM-DD'] + 'T' + data['HR:MN:SS.SS']
        data['time (datetime64)'] = np.array(times.tolist(), dtype = 'datetime64')
        high_rez.append(data)
        dates.append(data['time (datetime64)'].iloc[0].date())
print('high rez loaded')    

In [ ]:
## make file of 15 day velocity, rms of velocity fit, position mean and median, position std, and n
## this is fast as long as you don't care about rms of velocity. Very slow if you do. 
daily_data_smoothed2 = pd.DataFrame(columns=['velocity (m/d)', 'ste_velocity', 'std_velocity','rms_velocity', 'z_vel (m/d)',
                                            'time_mean','x_mean', 'y_mean', 'z_mean', 
                                            'time_median', 'x_median', 'y_median', 'z_median',
                                            'x_std', 'y_std', 'z_std', 'n'])

df_high_rez = pd.DataFrame({'date': dates, 'data': high_rez})
for idx,row in df_high_rez.iterrows():
    if np.mod(idx,100) == 0:
        print(str(idx) + ' of ' + str(df_high_rez.shape[0]))
    window_before = row['date'] - pd.Timedelta('7 days') # start of window is 7 days before
    window_after = row['date'] + pd.Timedelta('7 days') # end of window is 7 days after
    df_window = df_high_rez[df_high_rez['date'].between(window_before,window_after)] # grab data in window, inclusive

    # make a data frame of all the data in the window at full resolution
    window_data = pd.concat(df_window['data'].tolist(), ignore_index=True)

    # add a column to the dataframe for offset from the earliest day in days
    window_dt = window_data['time (datetime64)'] - window_data['time (datetime64)'].min()
    window_data['dt (days)'] = window_dt.dt.total_seconds()/(60*60*24)

    # calculate a best fit dLat (in degrees/day) and dLon (in degrees/day) for the time window (+/-7 days)
    p_lat, cov_lat = np.polyfit(window_data['dt (days)'].to_list(),window_data['lat_decimal']-window_data['lat_decimal'].mean(),1, cov=True)
    p_lon, cov_lon = np.polyfit(window_data['dt (days)'],window_data['lon_decimal']-window_data['lon_decimal'].mean(),1, cov=True)
    p_z = np.polyfit(window_data['dt (days)'],window_data['HGT(m)']-window_data['HGT(m)'].mean(),1)

    # calculate geodesic distance between the median coordinate - (dLon/2, dLat/2) 
    # and the median coordinate - (dLon/2,dLat/2), which gives you distance traveled/day, or v (m/day)
    lat_median = window_data['lat_decimal'].median()
    lon_median = window_data['lon_decimal'].median()
    pt1 = (lat_median - p_lat[0]/2, lon_median - p_lon[0]/2)
    pt2 = (lat_median + p_lat[0]/2, lon_median + p_lon[0]/2)
    v = geodesic(pt1,pt2).m # in m/day

    # calculate the standard error of the slope (velocity) fit by looking at the square root of
    # of the covariance of the latitude and longitude fits. 
    ste_lat_pt1 = (lat_median - np.sqrt(cov_lat[0][0]), lon_median)
    ste_lat_pt2 = (lat_median + np.sqrt(cov_lat[0][0]), lon_median)
    ste_lat = geodesic(ste_lat_pt1,ste_lat_pt2).m # std of latitude slope fit in meters
    ste_lon_pt1 = (lat_median, lon_median - np.sqrt(cov_lon[0][0]))
    ste_lon_pt2 = (lat_median, lon_median + np.sqrt(cov_lon[0][0]))
    ste_lon = geodesic(ste_lon_pt1,ste_lon_pt2).m # std of longitude slope fit in meters
    ste_velocity = np.sqrt(ste_lon**2+ste_lat**2) # propagate for std of velocity fit

    # calculate the uncertainty of velocity another way: randomly sample 90% of the data 
    # then recaculate the velocity and do this 100 times. This is obviously slow so set
    # std_velocity to np.nan if you are wanting to work quickly
    num_iter = 100
    v_estimates = []
    for idx in range(num_iter):
        this_sample = window_data.sample(int(len(window_data)*.9))
        this_p_lat, this_cov_lat = np.polyfit(this_sample['dt (days)'].to_list(),
                                    this_sample['lat_decimal']-this_sample['lat_decimal'].mean(),
                                    1, cov=True)
        this_p_lon, this_cov_lon = np.polyfit(this_sample['dt (days)'],
                                    this_sample['lon_decimal']-this_sample['lon_decimal'].mean(),
                                    1, cov=True)
        this_pt1 = (lat_median - this_p_lat[0]/2, lon_median - this_p_lon[0]/2)
        this_pt2 = (lat_median + this_p_lat[0]/2, lon_median + this_p_lon[0]/2)
        v_estimates.append(geodesic(this_pt1,this_pt2).m)
    std_velocity = np.std(v_estimates)
    # std_velocity = np.nan
    

    #### calculate the rms of the fit as a goodness of fit metric (which basically reflects position noise
    #### and is not a realistic "error" of velocity, but is worth calculating nonetheless)
    #### the geodesic distance calculation slows the code down substantially (down to ~ 1 second per 
    #### day, so we comment this out and set v_rms to np.nan if you are wanting to work quickly)
    lat_hats = np.polyval(p_lat,window_data['dt (days)'])+window_data['lat_decimal'].mean()
    lon_hats = np.polyval(p_lon,window_data['dt (days)'])+window_data['lon_decimal'].mean()
    e_fit = list(map(lambda allpts: geodesic((allpts[0],allpts[1]),(allpts[2],allpts[3])).m, 
                     zip(window_data['lat_decimal'].to_list(), window_data['lon_decimal'].to_list(),lat_hats, lon_hats)))
    v_rms = np.sqrt(np.mean(np.array(e_fit)**2)) #calc root mean squared
    # v_rms = np.nan

    #### calculate the std of v by 

    
    this_daily_smoothed = {'velocity (m/d)': [v],
                           'rms_velocity': [v_rms], 
                           'ste_velocity': [ste_velocity],
                           'std_velocity': [std_velocity],
                           'time_mean': [window_data['time (datetime64)'].mean()],
                           'z_vel (m/d)': [p_z[0]],
                           'x_mean': [np.mean(window_data['x'])],
                           'y_mean': [np.mean(window_data['y'])],
                           'z_mean': [np.mean(window_data['HGT(m)'])],
                           'time_median': [window_data['time (datetime64)'].median()],
                           'x_median': [np.median(window_data['x'])],
                           'y_median': [np.median(window_data['y'])],
                           'z_median': [np.median(window_data['HGT(m)'])],
                           'x_std': [np.std(window_data['x'])],
                           'y_std': [np.std(window_data['y'])],
                           'z_std': [np.std(window_data['HGT(m)'])],
                           'n': [window_data.shape[0]]}
    daily_data_smoothed2 = pd.concat([daily_data_smoothed2, pd.DataFrame(this_daily_smoothed)], ignore_index = True)

#daily_data_smoothed.to_pickle('tylg_velocity.pkl')
print('done')

In [ ]:
# save the daily data to csv for Hilary to plot however she wants
daily_data_smoothed2.to_csv('tylg_daily_15day-v2.csv')

In [ ]:
#plot some figures of the data you just produced
dates_of_interest = [np.datetime64('2018-09-10')]
xlimits = [np.datetime64('2017-11-09')+pd.Timedelta('3 days'), np.datetime64('2019-05-01')-pd.Timedelta('3 days')]
#xlimits = [dates_of_interest[0]-pd.Timedelta('200 days'), dates_of_interest[0]+pd.Timedelta('200 days')]

fig, [ax1,ax2] = plt.subplots(2,1,figsize=(6, 9))

#### plot z in subplot 1
ax1.plot(daily_data_smoothed2['time_mean'], daily_data_smoothed2['z_mean'],'o', label = 'mean')
ax1.plot(daily_data_smoothed2['time_mean'], daily_data_smoothed2['z_median'],'o', label = 'median')
ylimits=[86.12, 86.25] # tight limits

for dd in dates_of_interest:
    ax1.plot([dd, dd],[ylimits[0], ylimits[1]],'k-')
    ax1.text(dd+pd.Timedelta('3 days'),ylimits[0]+(ylimits[1]-ylimits[0])*0.03,
            dd.astype(datetime.date).strftime('%Y-%m-%d'),
            rotation=90)

ax1.set_xlim(xlimits)
ax1.set_ylim(ylimits)
ax1.legend()
ax1.set_title('TYLG')
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_minor_locator(mdates.MonthLocator())
ax1.set_ylabel('data points per day')


#### plot velocity
ylimits = [0.011, 0.015]
ax2.plot(daily_data_smoothed2['time_mean'], daily_data_smoothed2['velocity (m/d)'],'o')
#ylimits = [-0.0025, 0.0025]
#ax2.plot(daily_data_smoothed2['time_mean'], daily_data_smoothed2['z_vel (m/d)'],'o')

for dd in dates_of_interest:
    ax2.plot([dd, dd],[ylimits[0], ylimits[1]],'k-')
    ax2.text(dd+pd.Timedelta('3 days'),ylimits[0]+(ylimits[1]-ylimits[0])*0.03,
            dd.astype(datetime.date).strftime('%Y-%m-%d'),
            rotation=90)

ax2.set_xlim(xlimits)
ax2.set_ylim(ylimits)
ax2.set_xlabel('time')
ax2.set_ylabel('speed [m/day]')
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_minor_locator(mdates.MonthLocator())
plt.savefig('tylg_updated.png')
plt.show()